# Etude des données du fichier excel

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import json


On commence par load le json du fichier excel

In [11]:
file = Path("../data/raw_extracts/seed_data_20260416_170444.json")
data = json.loads(file.read_text(encoding="utf-8"))
df = pd.DataFrame(data)  # Remplacez "inventaire"
display(df)

,inventaire
0,"{'Unnamed: 0': None, 'Unnamed: 1': None, 'Unna..."
1,"{'Unnamed: 0': 'SERVICE', 'Unnamed: 1': 'EMPLA..."
2,"{'Unnamed: 0': '?????????', 'Unnamed: 1': '???..."
3,"{'Unnamed: 0': '?????????', 'Unnamed: 1': '???..."
4,"{'Unnamed: 0': '?????????', 'Unnamed: 1': '???..."
...,...
667,"{'Unnamed: 0': 'STOCK PC ', 'Unnamed: 1': 'DSI..."
668,"{'Unnamed: 0': 'STOCK PC ', 'Unnamed: 1': 'DSI..."
669,"{'Unnamed: 0': 'STOCK PC ', 'Unnamed: 1': 'DSI..."
670,"{'Unnamed: 0': 'STOCK PC ', 'Unnamed: 1': 'DSI..."


In [ ]:
df.describe(include="all")

,SERVICE,EMPLACEMENT,UTILISATEUR PRINCIPAL,NOM RESEAU,N° SERIE PC,CLEF WIFI,LECTEUR DVD/CD EXTERNE,WEBCAM,CASQUE,TYPE,...,DATE ACHAT ECRAN TELETRAVAIL,DATE FIN GARANTIE ECRAN TELETRAVAIL,TYPE ECRAN TELETRAVAIL2,MARQUE ECRAN TELETRAVAIL3,MODELE ECRAN TELETRAVAIL4,N° SERIE ECRAN TELETRAVAIL5,FOURNISSEUR ECRAN TELETRAVAIL6,N° BC ECRAN TELETRAVAIL7,DATE ACHAT ECRAN TELETRAVAIL8,DATE FIN GARANTIE ECRAN TELETRAVAIL2
count,670,665,530,487,515,4,16,26,29,628,...,1,1,1,1,1,1,1.0,1,1,0
unique,69,121,359,452,509,3,1,13,12,24,...,1,1,1,1,1,1,NaN,1,1,0
top,STOCK ECRAN,DSI,FOND DE CLASSE,LINUX,8YCL7S3,INTEGRE,OUI 22/11/2023,OUI,OUI,PC FIXE,...,2024-07-17 00:00:00,2029-07-16 00:00:00,DELL,DELL 27 - P2724DEB,8362K04,DELL,NaN,2024-07-31 00:00:00,2029-07-30 00:00:00,NaN
freq,129,204,25,24,2,2,16,12,14,180,...,1,1,1,1,1,1,NaN,1,1,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,32240163.0,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,32240163.0,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,32240163.0,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,32240163.0,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,32240163.0,NaN,NaN,NaN


Ici, on étudie le seuil de remplissage des données

In [ ]:
seuil = len(df) * 0.5
colonnes_vides = sorted(df.columns, key=lambda c: df[c].isna().sum(), reverse=True)

for col in colonnes_vides:
    if df[col].isna().sum() > seuil:
        print(f"Colonne '{col}' a plus de 50% de valeurs manquantes ({df[col].isna().sum()/len(df)*100:.2f} %)")

Colonne 'DATE FIN GARANTIE ECRAN TELETRAVAIL2' a plus de 50% de valeurs manquantes (100.00 %)
Colonne 'DATE ACHAT ECRAN TELETRAVAIL' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'DATE FIN GARANTIE ECRAN TELETRAVAIL' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'TYPE ECRAN TELETRAVAIL2' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'MARQUE ECRAN TELETRAVAIL3' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'MODELE ECRAN TELETRAVAIL4' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'N° SERIE ECRAN TELETRAVAIL5' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'FOURNISSEUR ECRAN TELETRAVAIL6' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'N° BC ECRAN TELETRAVAIL7' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'DATE ACHAT ECRAN TELETRAVAIL8' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'DATE ACHAT ECRAN 3' a plus de 50% de valeurs manquantes (99.70 %)
Colonne 'FOURNISSEUR ECRAN TELETRAVAIL' a plus de 50% de valeurs manquan

Il faut diviser le dataframe en 3 sous catégories : les colonnes à identifiants unique (comme addresse IP, addresse MAC, numéro de série etc), les catégories (type d'écran, marque, fournisseur) et les dates

Identifiants unique : Utilisateur principal, Nom réseau, n° série PC, mac adresse ethernet, mac adresse wifi, adresse IP PC, clef OS PC, clef activation, clef office

Catégories : Service, Emplacement, Clef wifi, Lecteur DVD/CD externe, Webcam, Casque, Type, Marque PC, Fournisseur PC, n° BC PC, eligible W10, OS, type licence PC, processeur PC, ram PC, Absolute Dell, Fournisseur Office, Version Office, Type licence Office, n° contrat open microsoft

Dates : date fin garantie pc, date achat pc, date achat office, date activation

puis une catégorie écrans pour toutes les colonnes d'écrans qui représentent la moitié du dataframe

In [12]:
df.isna().sum()

inventaire    0
dtype: int64

In [ ]:
identifiants = ['N° SERIE PC', 'NOM RESEAU', 'UTILISATEUR PRINCIPAL', 'MAC ADRESSE ETHERNET', 'MAC ADRESSE WIFI', 'Adresse IP PC', 'Clé OS PC', 'CLEF ACTIVATION', 'CLEF OFFICE']
categories = ['SERVICE', 'EMPLACEMENT', 'MARQUE PC', 'OS', 'RAM PC', 'VERSION OFFICE', 'OS', 'CLEF WIFI', 'LECTEUR DVD/CD EXTERNE', 'WEBCAM', 'CASQUE', 'TYPE', 'MARQUE PC', 'FOURNISSEUR PC', 'N° BC PC', 'eligible W10','TYPE LICENCE PC','PROCESSEUR PC', 'RAM PC', 'ABSOLUTE DELL', 'FOURNISSEUR OFFICE', 'VERSION OFFICE', 'TYPE LICENCE OFFICE', 'N° CONTRAT OPEN MICROSOFT']
dates = ['DATE ACHAT PC', 'DATE FIN GARANTIE PC', 'DATE ACHAT OFFICE', 'DATE ACTIVATION']
df_ecrans = df.filter(like='ECRAN')

In [ ]:
# A. Traitement des IDENTIFIANTS (Suppression des espaces inutiles)
for col in [c for c in identifiants if c in df.columns]:
    df[col] = df[col].astype(str).str.strip().str.upper()
display(df[identifiants].head())

,N° SERIE PC,NOM RESEAU,UTILISATEUR PRINCIPAL,MAC ADRESSE ETHERNET,MAC ADRESSE WIFI,Adresse IP PC,Clé OS PC,CLEF ACTIVATION,CLEF OFFICE
0,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE
1,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE
2,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE
3,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE
4,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE


In [ ]:
# B. Traitement des CATÉGORIES (Normalisation pour éviter les doublons "Dell" / "dell")
for col in [c for c in categories if c in df.columns]:
    df[col] = df[col].astype(str).str.strip().capitalize()
    # On remplace les "Nan" textuels par de vrais nuls
    df[col] = df[col].replace(['Nan', 'None', ''], pd.NA)

AttributeError: 'Series' object has no attribute 'capitalize'

On se penche en particulier sur les colonnes écrans

In [ ]:
df_ecrans = df.filter(like='ECRAN')

seuil = len(df_ecrans) * 0.5
colonnes_vides = sorted(df_ecrans.columns, key=lambda c: df_ecrans[c].isna().sum(), reverse=True)

for col in colonnes_vides:
    if df_ecrans[col].isna().sum() > seuil:
        print(f"Colonne '{col}' a plus de 50% de valeurs manquantes ({df_ecrans[col].isna().sum()/len(df_ecrans)*100:.2f} %)")
    else:
        print(f"Colonne '{col}' a moins de 50% de valeurs manquantes ({df_ecrans[col].isna().sum()/len(df_ecrans)*100:.2f} %)")

Colonne 'DATE FIN GARANTIE ECRAN TELETRAVAIL2' a plus de 50% de valeurs manquantes (100.00 %)
Colonne 'DATE ACHAT ECRAN TELETRAVAIL' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'DATE FIN GARANTIE ECRAN TELETRAVAIL' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'TYPE ECRAN TELETRAVAIL2' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'MARQUE ECRAN TELETRAVAIL3' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'MODELE ECRAN TELETRAVAIL4' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'N° SERIE ECRAN TELETRAVAIL5' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'FOURNISSEUR ECRAN TELETRAVAIL6' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'N° BC ECRAN TELETRAVAIL7' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'DATE ACHAT ECRAN TELETRAVAIL8' a plus de 50% de valeurs manquantes (99.85 %)
Colonne 'DATE ACHAT ECRAN 3' a plus de 50% de valeurs manquantes (99.70 %)
Colonne 'FOURNISSEUR ECRAN TELETRAVAIL' a plus de 50% de valeurs manquan

In [ ]:
# 1. On définit les colonnes qui identifient le PC
id_vars = ['TYPE']

# 2. On isole les colonnes de marques (on peut faire pareil pour Modèle, Série, etc.)
cols_marques = [c for c in df.columns if 'MARQUE ECRAN' in c]

df_marques_ecrans = df.melt(
    id_vars=id_vars, 
    value_vars=cols_marques,
    var_name='SLOT', 
    value_name='MARQUE'
)

df_marques_ecrans = df_marques_ecrans.dropna(subset=['MARQUE'])

display(df_marques_ecrans.head())

#df_marques_ecrans['POSITION_ECRAN'] = df_marques_ecrans['POSITION_ECRAN'].str.extract('(\d+)')

,N° SERIE PC,SERVICE,UTILISATEUR PRINCIPAL,POSITION_ECRAN,MARQUE
0,NONE,?????????,NONE,MARQUE ECRAN 1,DELL
1,NONE,?????????,NONE,MARQUE ECRAN 1,DELL
2,NONE,?????????,NONE,MARQUE ECRAN 1,DELL
3,NONE,?????????,NONE,MARQUE ECRAN 1,LENOVO
4,NONE,?????????,NONE,MARQUE ECRAN 1,PHILIPS


In [ ]:
col = "PROCESSEUR PC"  # ← change ce nom pour naviguer
print(f"=== {col} ===")
for val in df[col].dropna().unique():
    print(val)

=== PROCESSEUR PC ===
Intel(R) Core(TM) i5-10500 (6 Cores/12MB/12T/3.1GHz to 4.5GHz/65W)
Intel® Core™ i5-1235U de 12e génération (10 cœurs - 4,40 GHz)
Intel(R) Core(TM) i5-8500 CPU @ 3.00GHz
Intel(R) Core(TM) i5-9500
12th Generation Intel Core i5-12500T
Intel Core i5-4590 3,3 Ghz
I5-1135G7
11th Generation Intel Core i5-1135G7
Intel(R) Core(TM) i5-9500 CPU @ 3.00GHz
Intel®Core™ i7-1250U (12Mo Cache, jusqu’à 4,7 GHz, 10 Coeurs)
Intel(R) Core(TM) i5-10505 CPU @ 3.20GHz
Intel I5 - 512 GB SSD
Intel Core i5-10505 (6 coeurs/12Mo/12T/de 3,2GHz à 4,6GHz/65W)
Intel(R) Core(TM) i7-7700 CPU @ 3.60GHz
Intel® Core™ Ultra 5 235U vPro™ (12 TOPS NPU, 12 coeurs, 4,9 GHz)
Intel® Core™ Ultra 7 255H (24Mo, 16 Coeurs, 5,10GHz Turbo, 45W)
Processeur Intel Core Ultra 7 155H, vPro Essentials (24 mo 16 CŒURS)
Core 2 DUO P7350@2Ghz
i7-2670QM
i5-6200U
Intel Core i5-8250U 
Intel i5-1135G7
Intel® Core™ Ultra 5 125U processeur - 12 Cœurs - 4,3 GHz
Intel i5-1135G7, Intel Iris Xe Graphics Capable
Intel(R) Core(TM) i5-

In [ ]:
colonnes_triees = sorted(df.columns, key=lambda col: df[col].nunique(), reverse=True)

for col in colonnes_triees:
    print(f"{col}: {df[col].nunique()}")

N° SERIE PC: 509
NOM RESEAU: 452
MAC ADRESSE ETHERNET: 445
N° SERIE ECRAN 1: 435
UTILISATEUR PRINCIPAL: 359
MAC ADRESSE WIFI: 184
DATE ACTIVATION: 183
CLEF OFFICE: 170
MODELE ECRAN 1: 141
EMPLACEMENT: 121
CLEF ACTIVATION: 110
DATE ACHAT PC: 93
N° BC PC: 86
Clé OS PC: 76
DATE FIN GARANTIE PC: 73
SERVICE: 69
PROCESSEUR PC: 66
MARQUE PC: 62
N° SERIE ECRAN 2: 59
DATE ACHAT ECRAN 1: 53
N° BC ECRAN 1: 48
DATE ACHAT OFFICE: 33
TYPE ECRAN 1: 33
VERSION OFFICE: 32
BC OFFICE: 31
MODELE ECRAN 2: 30
MARQUE ECRAN 1: 29
DATE FIN GARANTIE ECRAN 1: 26
TYPE: 24
N° BC ECRAN 2: 22
DATE ACHAT ECRAN 2: 22
Mail ACTIVATION: 18
Adresse IP PC: 17
OS: 15
FOURNISSEUR ECRAN 1: 14
WEBCAM: 13
N° CONTRAT OPEN MICROSOFT: 13
CASQUE: 12
TYPE ECRAN 2: 12
FOURNISSEUR PC: 11
RAM PC: 10
DATE FIN GARANTIE ECRAN 2: 10
MARQUE ECRAN 2: 9
FOURNISSEUR OFFICE: 8
FOURNISSEUR ECRAN 2: 7
TYPE LICENCE OFFICE: 6
TYPE ECRAN TELETRAVAIL: 6
eligible W10: 5
N° SERIE ECRAN TELETRAVAIL: 4
CLEF WIFI: 3
TYPE LICENCE PC: 3
TYPE ECRAN 3: 3
MARQ

In [ ]:
# 1. Identifier les colonnes qui respectent la condition (< 20 valeurs uniques, hors NaN)
colonnes_filtrees = [col for col in df.columns if df[col].nunique() < 20]

# 2. Créer le nouveau DataFrame avec uniquement ces colonnes
df_reduit = df[colonnes_filtrees].copy()

# Affichage pour vérifier
df_reduit.head()

,CLEF WIFI,LECTEUR DVD/CD EXTERNE,WEBCAM,CASQUE,FOURNISSEUR PC,Adresse IP PC,eligible W10,OS,TYPE LICENCE PC,RAM PC,...,DATE ACHAT ECRAN TELETRAVAIL,DATE FIN GARANTIE ECRAN TELETRAVAIL,TYPE ECRAN TELETRAVAIL2,MARQUE ECRAN TELETRAVAIL3,MODELE ECRAN TELETRAVAIL4,N° SERIE ECRAN TELETRAVAIL5,FOURNISSEUR ECRAN TELETRAVAIL6,N° BC ECRAN TELETRAVAIL7,DATE ACHAT ECRAN TELETRAVAIL8,DATE FIN GARANTIE ECRAN TELETRAVAIL2
0,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,NaN,None,None,None
1,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,NaN,None,None,None
2,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,NaN,None,None,None
3,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,NaN,None,None,None
4,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,NaN,None,None,None


In [ ]:
seuil = len(df) * 0.5
df = df.dropna(thresh=seuil, axis=1)